# Nano_ext — Interactive Analysis Notebook

This notebook provides an interactive UI for detecting ionic current blockade events from nanopore current traces.

## Setup (first time only)

```bash
conda activate nano_ext
pip install "nano_ext[notebook]"
jupyter notebook
```

## How to use

1. **Run cells from top to bottom** (Shift+Enter)
2. Once the UI appears, work through each section in order:
   - **1. File Loading** — select your recording file (and optional control file) via the file browser, then click `Load`
   - **2. Analysis Range** *(collapsed, optional)* — restrict analysis to a time window (`Keep range`) or exclude artifact regions such as zapping (`Exclude ranges`); use the `Preview signal` button to verify the selection
   - **3. Analysis Settings** — choose event direction (Down / Up / Both) and enable Auto-tune
   - **4. Additional Settings** *(collapsed)* — software filter toggle and advanced detection parameters, each with explanations
   - **5. Run** — click `Run Analysis`; a live progress log appears while the pipeline runs
   - **6. Results** — interactive waveform with baseline/threshold overlays, event table, and `Download CSV` button

---

In [ ]:
# Check that all required packages are available
try:
    import nano_ext
    import ipywidgets
    import plotly
    print(f"nano_ext   {nano_ext.__version__}")
    print(f"ipywidgets {ipywidgets.__version__}")
    print(f"plotly     {plotly.__version__}")
    print("All packages present.")
except ImportError as e:
    print(f"Missing package: {e}")
    print('Run:  pip install "nano_ext[notebook]"')

In [ ]:
from nano_ext.outputs.notebook_widgets import NanoExtUI

ui = NanoExtUI()
ui.display()

---

## Custom downstream analysis

After clicking **Run Analysis**, the full `PipelineResult` is accessible at `ui._result`.
The cells below show common starting points — run them after the analysis completes.

In [ ]:
# Build a pandas DataFrame from the detected events
import pandas as pd

result = ui._result
if result is None:
    print("No analysis run yet. Click 'Run Analysis' in the UI above first.")
else:
    sr = result.signal_data.sampling_rate
    records = [
        {
            "event_id": i + 1,
            "start_sec": ev.start_idx / sr,
            "end_sec": ev.end_idx / sr,
            "duration_ms": ev.duration * 1000,
            "depth": ev.depth,
            "area": ev.area,
            "n_sublevels": len(ev.sublevels),
            "direction": ev.direction.value,
        }
        for i, ev in enumerate(result.events)
    ]
    df = pd.DataFrame(records)
    print(f"{len(df)} events detected")
    df.head()

In [ ]:
# Histogram of event depths
if result is not None and result.events:
    import plotly.express as px
    fig = px.histogram(
        df, x="depth",
        nbins=50,
        title="Distribution of event depths",
        labels={"depth": f"Depth ({result.signal_data.units})"},
    )
    fig.show()

In [ ]:
# Histogram of event durations
if result is not None and result.events:
    fig = px.histogram(
        df, x="duration_ms",
        nbins=50,
        title="Distribution of event durations",
        labels={"duration_ms": "Duration (ms)"},
    )
    fig.show()